# Chapter 6 gallery, robust multivariate analysis

Reproduces `biochem.R` (Ex 6.1), `vehicle.R` (Ex 6.3), `bus.R` (Ex 6.4) and the `covRobMM` part of `wine1.R` (Ex 6.5–6.6).

In [ ]:
import os, sys, pathlib


import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")  # headless-safe for CI execution
import matplotlib.pyplot as plt
import robstattm_py as rpm
from robstattm_py import set_seed
from robstattm_py._r import r as _r

ro = _r()
ro.r("suppressMessages(library(RobStatTM))")
FIG_DIR = pathlib.Path("figures"); FIG_DIR.mkdir(exist_ok=True)
print(f"robstattm_py {rpm.__version__}")

## biochem, classical location/scatter motivation (Example 6.1)

`biochem.R` uses only classical statistics (mean, var, correlation) to show how a single observation (#3) distorts them, motivating the robust estimators in the rest of the chapter. No RobStatTM estimator is called here; it is the classical baseline.

In [ ]:
import numpy as np
biochem = rpm.datasets.biochem()
X = biochem.to_numpy(dtype=float)
def stats(M):
    mu = M.mean(axis=0); v = M.var(axis=0, ddof=1)
    rho = np.corrcoef(M, rowvar=False)[0, 1]
    return mu, v, rho
mu, v, rho = stats(X)
mu2, v2, rho2 = stats(np.delete(X, 2, axis=0))  # drop obs 3 (0-based 2)
print('with obs 3 :  means', np.round(mu,2), 'vars', np.round(v,2), 'rho', round(rho,2))
print('drop obs 3 :  means', np.round(mu2,2), 'vars', np.round(v2,2), 'rho', round(rho2,2))
print('=> one point flips the correlation: classical stats are not robust')

## vehicle, Rocke S-estimator vs classical distances (Example 6.3)

`vehicle.R` compares classical and `covRobRocke` Mahalanobis distances on the vehicle-silhouette data (217×18). The robust distances expose outliers masked by the classical covariance. (The rrcov CovMcd/CovSest comparators in the script are out of scope and omitted.)

In [ ]:
vehicle = rpm.datasets.vehicle()
Xv = vehicle.to_numpy(dtype=float)
n, p = Xv.shape
# classical Mahalanobis distances
xbar = Xv.mean(axis=0); C = np.cov(Xv, rowvar=False)
diff = Xv - xbar
disC = np.einsum('ij,jk,ik->i', diff, np.linalg.inv(C), diff)
# Rocke robust distances (stochastic -> seed for R parity)
set_seed(1)
rk = rpm.cov_rob_rocke(Xv)
print(rk)
print('robust vs classical max distance:', round(rk.dist.max(),1), 'vs', round(disC.max(),1))

In [ ]:
from scipy.stats import chi2
qua = chi2.ppf((np.arange(1, n+1) - 0.5) / n, p)
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, d, title in ((axes[0], np.sort(disC), 'Classical'),
                     (axes[1], np.sort(rk.dist), 'Rocke')):
    ax.scatter(qua, d, c='k', s=12); ax.axline((0,0), slope=1, color='r')
    ax.set_title(title); ax.set_xlabel('chi-square quantiles'); ax.set_ylabel('sorted distances')
fig.savefig(FIG_DIR / 'ch6_vehicle.png', dpi=110, bbox_inches='tight'); plt.close(fig)
print('Fig 6.7 analogue saved')

### Strict-tier cross-check vs direct R `covRobRocke`

In [ ]:
ro.globalenv['Xv'] = Xv
set_seed(1)
rk = rpm.cov_rob_rocke(Xv)
ro.r('set.seed(1L); rr <- covRobRocke(Xv)')
print('center bit-equal to R:', np.array_equal(rk.center, np.asarray(ro.r('as.numeric(rr$center)'), dtype=float)))
print('cov    bit-equal to R:', np.array_equal(rk.cov, np.asarray(ro.r('rr$cov'), dtype=float)))

## bus, robust PCA via M-scale (Example 6.4)

`bus.R` drops column 9, standardizes by median/MAD, then compares classical PCA reconstruction error to `pcaRobS` (3 components).

In [ ]:
bus = rpm.datasets.bus()
X0 = bus.to_numpy(dtype=float)
X1 = np.delete(X0, 8, axis=1)  # drop column 9 (0-based 8)
from scipy.stats import median_abs_deviation
med = np.median(X1, axis=0)
mads = median_abs_deviation(X1, axis=0, scale='normal')
Xb = (X1 - med) / mads
set_seed(42)
rr = rpm.pca_rob_s(Xb, ncomp=3)
print(rr)
print('proportion of robust scale explained (3 comps):', round(float(rr.propex), 4))
# residual reconstruction distances, robust vs classical
resiM = Xb - rr.fit
dM = (resiM**2).sum(axis=1)
print('robust reconstruction: max row distance =', round(dM.max(), 2))

### Strict-tier cross-check vs direct R `pcaRobS`

In [ ]:
ro.globalenv['Xb'] = Xb
set_seed(42)
rr = rpm.pca_rob_s(Xb, ncomp=3)
ro.r('set.seed(42L); rp <- pcaRobS(Xb, ncomp=3)')
print('mu  bit-equal to R:', np.array_equal(rr.mu, np.asarray(ro.r('as.numeric(rp$mu)'), dtype=float)))
print('fit bit-equal to R:', np.array_equal(rr.fit, np.asarray(ro.r('rp$fit'), dtype=float)))

## wine1, MM covariance under independent contamination (Examples 6.5–6.6)

`wine1.R` flags multivariate outliers with `covRobMM` distances on the wine data. (The missing-data and cell-wise parts of `wine1.R` use `GSE::GSE` / `GSE::TSGS`, those are reproduced in `notebooks/external_demo.ipynb`.)

In [ ]:
wine = rpm.datasets.wine()
Xw = wine.to_numpy(dtype=float)
n, p = Xw.shape
from scipy.stats import chi2
qq = chi2.ppf(0.999, p)
set_seed(100)
mm = rpm.cov_rob_mm(Xw)
flagged = np.flatnonzero(mm.dist > qq)
print(mm)
print(f'rows with robust distance > chi2(.999, {p}) = {qq:.1f}: {(flagged+1).tolist()}')

In [ ]:
ro.globalenv['Xw'] = Xw
set_seed(100)
mm = rpm.cov_rob_mm(Xw)
ro.r('set.seed(100L); rm <- covRobMM(Xw)')
print('center bit-equal to R:', np.array_equal(mm.center, np.asarray(ro.r('as.numeric(rm$center)'), dtype=float)))
print('dist   bit-equal to R:', np.array_equal(mm.dist, np.asarray(ro.r('as.numeric(rm$dist)'), dtype=float)))